# AutoEIT Automated Scoring System
## GSoC 2026 — HumanAI Foundation

This notebook implements an automated scoring system for the **Elicited Imitation Task (EIT)** — 
a research tool used to measure global language proficiency in second language learners.

### Approach
The scoring pipeline uses a **3-layer hybrid system**:

**Layer 1 — Rule-Based Scorer (transparent baseline)**
- Normalizes text by removing accents and punctuation
- Filters disfluencies and filler words (`um`, `uh`, `...`)
- Matches words using fuzzy matching (`rapidfuzz`) to handle spelling variations
- Uses Spanish stemming (`nltk SnowballStemmer`) to handle verb form differences
- Maps word match percentage to 0-4 scale

**Layer 2 — Semantic Similarity (ML signal)**
- Encodes stimulus and transcription using `paraphrase-multilingual-MiniLM-L12-v2`
- Computes cosine similarity in vector space
- Captures meaning preservation even when words differ completely

**Layer 3 — LLM Scorer (edge case handler)**
- Uses Groq (Llama 3.1 8b) with `temperature=0.0` for reproducible scoring
- Applies meaning-based rubric with Spanish linguistic understanding
- Handles disfluent speech, partial sentences, and mixed language
- Provides human-readable reasoning for every score

### Final Score Calculation
```
Final Score = (Rule Score × 0.4) + (LLM Score × 0.4) + (Similarity Score × 0.2)
```
- Rule-based and LLM get equal weight as primary scorers
- Semantic similarity acts as a supporting signal
- Score is clamped to 0-4 range

### Why This Approach
- **Transparency** — rule-based layer makes every scoring decision explainable
- **Consistency** — `temperature=0.0` ensures identical LLM scores on repeated runs, directly addressing the inconsistency problem stated in the AutoEIT2 proposal
- **Robustness** — 3 independent signals reduce overconfidence in edge cases
- **Multilingual** — all components handle Spanish natively

### Scoring Rubric (0-4 scale)
| Score | Meaning |
|---|---|
| 0 | No meaningful words reproduced, meaning completely lost |
| 1 | Very few words correct, meaning mostly lost |
| 2 | Some words correct, partial meaning preserved |
| 3 | Most words correct, meaning mostly preserved |
| 4 | All or nearly all words correct, meaning fully preserved |

In [1]:
import os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from dotenv import load_dotenv
load_dotenv()

from preprocessor import load_all_files
from scorer import score_participant

print("All imports successful")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2689.20it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All imports successful


In [2]:
import glob

# Exclude the Info file
FILES = sorted([f for f in glob.glob('data/*.csv') if 'Info' not in f])

print(f"Found {len(FILES)} participant files:")
for f in FILES:
    print(f"  {f}")

sample = pd.read_csv(FILES[0], encoding='latin-1')
print(f"\nColumns found: {sample.columns.tolist()}")

Found 4 participant files:
  data\AutoEIT Sample Transcriptions for Scoring(38001-1A).csv
  data\AutoEIT Sample Transcriptions for Scoring(38002-2A).csv
  data\AutoEIT Sample Transcriptions for Scoring(38004-2A).csv
  data\AutoEIT Sample Transcriptions for Scoring(38006-2A).csv

Columns found: ['Sentence', 'Stimulus', 'Transcription Rater 1', 'Score']


In [3]:
dataframes = load_all_files(FILES)

print(f"\nLoaded {len(dataframes)} participant files")
print(f"Sentences per participant: {len(dataframes[0])}")
print("\nSample data — Participant 1:")
dataframes[0][['Sentence', 'Stimulus', 'Transcription Rater 1']].head(5)


Loaded 4 participant files
Sentences per participant: 30

Sample data — Participant 1:


,Sentence,Stimulus,Transcription Rater 1
0,1.0,Quiero cortarme el pelo,Quiero cortarme el pelo
1,2.0,El libro está en la mesa,El libro está en la mesa
2,3.0,El carro lo tiene Pedro,El carro lo tiene Pedro
3,4.0,El se ducha cada mañana,El se ducha cada mañana
4,5.0,¿Qué dice usted que va a hacer hoy?,Que dices ustedes se que van a hacer hoy?


## Example Scoring Walkthrough

Before running the full pipeline, let's see how a single sentence is scored.

The scorer:
1. Sends the stimulus and transcription to the LLM with the rubric
2. Computes semantic similarity using multilingual embeddings
3. Combines both into a final score

In [6]:
from scorer import score_utterance

# Pick one example
stimulus = "Dudo que sepa manejar muy bien"
transcription = "Dudo que sepa manajar bien"

result = score_utterance(stimulus, transcription)

print(f"Rule Score: {result['Rule_Score']}")
print(f"LLM_Score: {result['LLM_Score']}")
print(f"Semantic_Similarity: {result['Semantic_Similarity']}")
print(f"Score: {result['Score']}")
print(f"Divergence Flag: {result['Divergence_Flag']}")
print(f"LLM Reasoning: {result['Reasoning']}")

Rule Score: 3
LLM_Score: 3
Semantic_Similarity: 0.8548
Score: 3
Divergence Flag: False
LLM Reasoning: The learner reproduced most of the words correctly, with only a slight variation in the verb 'manejar' (manajar), but the overall meaning of the sentence is mostly preserved.


In [7]:
os.makedirs("output", exist_ok=True)

all_results = []

for df in dataframes:
    participant = df['Participant'].iloc[0]
    print(f"\nScoring: {participant}")
    
    scored_df = score_participant(df)
    all_results.append(scored_df)
    
    print(f"  Done. Total: {scored_df['Score'].sum()} / {len(scored_df) * 4}")

combined = pd.concat(all_results, ignore_index=True)
print("\n✅ All participants scored")

# Display results for each participant
for scored_df in all_results:
    participant = scored_df['Participant'].iloc[0]
    print(f"\n{'='*60}")
    print(f"Participant: {participant}")
    print(f"{'='*60}")
    display(scored_df[['Sentence', 'Stimulus', 'Transcription Rater 1', 
                 'Rule_Score', 'LLM_Score', 'Semantic_Similarity', 
                 'Score', 'Divergence_Flag', 'Reasoning']])


Scoring: AutoEIT Sample Transcriptions for Scoring(38001-1A)
  Scoring: Quiero cortarme el pelo...
  Scoring: El libro está en la mesa...
  Scoring: El carro lo tiene Pedro...
  Scoring: El se ducha cada mañana...
  Scoring: ¿Qué dice usted que va a hacer hoy?...
  Scoring: Dudo que sepa manejar muy bien...
  Scoring: Las calles de esta ciudad son muy anchas...
  Scoring: Puede que llueva mañana todo el día...
  Scoring: Las casas son muy bonitas pero caras...
  Scoring: Me gustan las películas que acaban bien...
  Scoring: El chico con el que yo salgo es español...
  Scoring: Después de cenar me fui a dormir tranquilo...
  Scoring: Quiero una casa en la que vivan mis animales...
  Scoring: A nosotros nos fascinan las fiestas grandiosas...
  Scoring: Ella sólo bebe cerveza y no come nada...
  Scoring: Me gustaría que el precio de las casas bajara...
  Scoring: Cruza a la derecha y después sigue todo recto...
  Scoring: Ella ha terminado de pintar su apartamento...
  Scoring: Me gustar

,Sentence,Stimulus,Transcription Rater 1,Rule_Score,LLM_Score,Semantic_Similarity,Score,Divergence_Flag,Reasoning
0,1.0,Quiero cortarme el pelo,Quiero cortarme el pelo,4,4,1.0000,4,False,The learner fully preserved the meaning of the...
1,2.0,El libro está en la mesa,El libro está en la mesa,4,4,1.0000,4,False,The learner reproduced the prompt sentence exa...
2,3.0,El carro lo tiene Pedro,El carro lo tiene Pedro,4,4,1.0000,4,False,The learner fully preserved the meaning of the...
3,4.0,El se ducha cada mañana,El se ducha cada mañana,4,4,1.0000,4,False,The learner fully preserved the meaning of the...
4,5.0,¿Qué dice usted que va a hacer hoy?,Que dices ustedes se que van a hacer hoy?,3,2,0.9547,3,False,"The learner reproduced some words correctly, s..."
5,6.0,Dudo que sepa manejar muy bien,Dudo que sepa manajar bien,3,3,0.8548,3,False,The learner reproduced most of the words corre...
6,7.0,Las calles de esta ciudad son muy anchas,Las calles de esta cuidad son muy anchas,4,3,0.9567,4,False,The learner reproduced most of the words corre...
7,8.0,Puede que llueva mañana todo el día,Puede que lleva mañana todo el día,4,3,0.7493,3,False,The learner reproduced most of the words corre...
8,9.0,Las casas son muy bonitas pero caras,Las casas son muy bonitas pero muy cadas,3,3,0.8731,3,False,The learner preserved the meaning of the origi...
9,10.0,Me gustan las películas que acaban bien,Me gustan las peliculas que acaban bien,4,3,0.5403,3,False,The learner reproduced most of the words corre...



Participant: AutoEIT Sample Transcriptions for Scoring(38002-2A)


,Sentence,Stimulus,Transcription Rater 1,Rule_Score,LLM_Score,Semantic_Similarity,Score,Divergence_Flag,Reasoning
0,1.0,Quiero cortarme el pelo,Quiero cortarme el pelo,4,4,1.0000,4,False,The learner fully preserved the meaning of the...
1,2.0,El libro está en la mesa,El libro está en la mesa.,4,4,0.9980,4,False,The learner reproduced the prompt sentence exa...
2,3.0,El carro lo tiene Pedro,El carro tiene pelo.,2,1,0.3285,1,False,The learner reproduced very few words correctl...
3,4.0,El se ducha cada mañana,él se ducha cada mañana,4,4,0.9909,4,False,The learner reproduced the prompt sentence wit...
4,5.0,¿Qué dice usted que va a hacer hoy?,en que ustedes x uf x hacer hoy?,2,2,0.8503,2,False,"The learner reproduced some words correctly, s..."
5,6.0,Dudo que sepa manejar muy bien,dudo/tu no? sepiar exx muy bien,1,2,0.5456,2,False,"The learner reproduced some words correctly, s..."
6,7.0,Las calles de esta ciudad son muy anchas,La calles estén xxxx muy anchas.,1,2,0.9280,2,False,"The learner reproduced some words correctly, s..."
7,8.0,Puede que llueva mañana todo el día,Puede que mañana todo la día,3,2,0.7646,3,False,"The learner reproduced some words correctly, s..."
8,9.0,Las casas son muy bonitas pero caras,Las casas muy bonitos pero ... casas,3,2,0.7801,3,False,The learner partially preserved the meaning by...
9,10.0,Me gustan las películas que acaban bien,Me gustan las películas campan bien,3,2,0.9547,3,False,"The learner reproduced some words correctly, s..."



Participant: AutoEIT Sample Transcriptions for Scoring(38004-2A)


,Sentence,Stimulus,Transcription Rater 1,Rule_Score,LLM_Score,Semantic_Similarity,Score,Divergence_Flag,Reasoning
0,1.0,Quiero cortarme el pelo,Quiero cortarme el pelo,4,4,1.0000,4,False,The learner fully preserved the meaning of the...
1,2.0,El libro está en la mesa,El libro está en la mesa,4,4,1.0000,4,False,The learner reproduced the prompt sentence exa...
2,3.0,El carro lo tiene Pedro,El carro tiene Pedro,3,3,0.9742,3,False,The learner preserved the meaning of the promp...
3,4.0,El se ducha cada mañana,El se ducha cada mañana,4,4,1.0000,4,False,The learner fully preserved the meaning of the...
4,5.0,¿Qué dice usted que va a hacer hoy?,¿Qué dices usted vas hacer hoy?,3,3,0.9888,3,False,The learner reproduced most of the words corre...
5,6.0,Dudo que sepa manejar muy bien,Dudo que sepa manejar muy bien,4,4,1.0000,4,False,The learner reproduced the prompt sentence exa...
6,7.0,Las calles de esta ciudad son muy anchas,Las calles de esta ciudad es muy anchas,3,3,0.9997,3,False,The learner reproduced most of the words corre...
7,8.0,Puede que llueva mañana todo el día,Pueda que llueva mañana todo día,3,3,0.9792,3,False,The learner reproduced most of the words corre...
8,9.0,Las casas son muy bonitas pero caras,Las casas son muy bonitas para caras,3,3,0.8431,3,False,The learner preserved the meaning of the origi...
9,10.0,Me gustan las películas que acaban bien,Mis gus..Me gustas las películas que cada bien,3,2,0.9162,3,False,"The learner reproduced some words correctly, s..."



Participant: AutoEIT Sample Transcriptions for Scoring(38006-2A)


,Sentence,Stimulus,Transcription Rater 1,Rule_Score,LLM_Score,Semantic_Similarity,Score,Divergence_Flag,Reasoning
0,1.0,Quiero cortarme el pelo,Quiero cortarme mi pelo,3,3,0.9965,3,False,The learner reproduced most of the words corre...
1,2.0,El libro está en la mesa,El libro [pause] está en la mesa,4,3,0.8965,4,False,The learner reproduced most of the words corre...
2,3.0,El carro lo tiene Pedro,E-[gibberish] perro,0,1,0.2750,1,False,"The learner reproduced very few words correct,..."
3,4.0,El se ducha cada mañana,El se lucha cada mañana,3,2,0.5281,2,False,"The learner reproduced some words correctly, b..."
4,5.0,¿Qué dice usted que va a hacer hoy?,¿Qué [gibberish] que vas estoy?,1,1,0.5088,1,False,The learner reproduced very few words correctl...
5,6.0,Dudo que sepa manejar muy bien,Dudo que sepa ma-mastar tan bien (tambien?),2,3,0.6070,2,False,The learner reproduced most of the words corre...
6,7.0,Las calles de esta ciudad son muy anchas,Las calles..es-[gibberish]...,0,1,0.5222,1,False,The learner reproduced very few words correctl...
7,8.0,Puede que llueva mañana todo el día,Puede xxx mañana de todo día,2,2,0.7224,2,False,"The learner reproduced some words correctly, s..."
8,9.0,Las casas son muy bonitas pero caras,A las casa es mu-son bonitas,1,2,0.6584,2,False,"The learner reproduced some words correctly, s..."
9,10.0,Me gustan las películas que acaban bien,Me gusta las películas que x bien,3,2,0.9567,3,False,"The learner reproduced some words correctly, s..."


In [8]:
print("===== SCORING SUMMARY =====\n")

summary_data = []

for scored_df in all_results:
    participant = scored_df['Participant'].iloc[0]
    total = scored_df['Score'].sum()
    maximum = len(scored_df) * 4
    pct = round((total / maximum) * 100, 1)
    avg_sim = scored_df['Semantic_Similarity'].mean()
    flagged = scored_df['Divergence_Flag'].sum()
    
    summary_data.append({
        'Participant': participant,
        'Score': f"{total}/{maximum}",
        'Percentage': f"{pct}%",
        'Avg Similarity': round(avg_sim, 4),
        'Divergent Sentences': flagged
    })

summary_df = pd.DataFrame(summary_data)
summary_df

===== SCORING SUMMARY =====



,Participant,Score,Percentage,Avg Similarity,Divergent Sentences
0,AutoEIT Sample Transcriptions for Scoring(3800...,96/120,80.0%,0.9159,0
1,AutoEIT Sample Transcriptions for Scoring(3800...,67/120,55.8%,0.7791,0
2,AutoEIT Sample Transcriptions for Scoring(3800...,87/120,72.5%,0.8843,0
3,AutoEIT Sample Transcriptions for Scoring(3800...,58/120,48.3%,0.6510,2


## Evaluation of Output

### How I validated the scores:
- **Consistency** — `temperature=0.0` ensures identical LLM scores on repeated runs, 
  directly addressing the inconsistency problem stated in the AutoEIT2 proposal
- **3-layer cross validation** — Rule-based, semantic similarity, and LLM scores are 
  computed independently. High divergence between layers flags sentences for human review
- **Fuzzy + stem matching** — Rule-based layer uses `rapidfuzz` and Spanish stemming 
  to handle spelling variations and verb forms, making word matching more accurate than 
  exact string comparison
- **Multilingual embeddings** — `paraphrase-multilingual-MiniLM-L12-v2` is specifically 
  trained for non-English languages, avoiding cross-lingual bias in similarity scoring

### Limitations:
- Without ground truth human scores, absolute accuracy cannot be measured — 
  divergence flagging is used as a proxy for uncertain cases
- Rule-based scorer may underperform on sentences with heavy paraphrasing or 
  synonyms since it relies on word-level matching
- LLM may occasionally misinterpret highly disfluent utterances with incomplete 
  sentence fragments
- Fuzzy matching threshold (80%) is heuristic — optimal threshold may vary 
  across proficiency levels
